# Numeric solver vs simulation

The Rust solver in `src/moran_process/numeric` returns, per starting node, the fixation
probability `u_i`, the unconditional absorption time, and the conditional fixation time.
The simulation starts the mutant at a uniformly random node, so the matching quantities are


Both count elementary birth-death events, null events included.
Run section 1 once, wait for the array, then run 2-4.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from moran_process.core.population_graph import PopulationGraph

from moran_process.pipeline.process_lab import ProcessLab

BATCH_NAME = "2026_09_22-numeric-vs-simulation"
R = 1.1
N_REPEATS = 200_000
N_JOBS = 9
BATCH_DIR = Path("simulation_data") / BATCH_NAME
SHARDS = str(BATCH_DIR / "tmp" / "results" / "*.parquet")

In [ ]:
GRAPHS = [
    # --- undirected ---
    PopulationGraph.mammalian_lung_graph(branching_factor=2, depth=3),
    # PopulationGraph.avian_graph(n_rods=6, rod_length=3),
    # PopulationGraph.fish_graph(n_rods=2, rod_length=2, fillaments=3),
    # PopulationGraph.complete_graph(21),
    # PopulationGraph.cycle_graph(21),
    # PopulationGraph.star_graph(21),
    # PopulationGraph.line_graph(21),
    # PopulationGraph.grid_graph(4, 5),
    # --- directed: offspring flow along edge direction only ---
    # PopulationGraph.avian_graph(n_rods=6, rod_length=3, directed=True),
]

pd.DataFrame(
    [(g.name, g.n_nodes, g.graph.number_of_edges()) for g in GRAPHS],
    columns=["name", "n_nodes", "n_edges"],
)

---
## 1. Simulate

`post_batch="none"`: section 3 scans the raw shards itself, so none of the rollup jobs are
needed. Run this cell once, then `bjobs -w`.

In [ ]:
(BATCH_DIR / "tmp").mkdir(parents=True, exist_ok=True)
zoo_path = BATCH_DIR / "tmp" / "graph_zoo.joblib"
joblib.dump(GRAPHS, zoo_path)

ProcessLab().submit_jobs(
    zoo_path=zoo_path,
    n_graphs=len(GRAPHS),
    r_values=[R],
    batch_name=BATCH_NAME,
    batch_dir=BATCH_DIR,
    n_repeats=N_REPEATS,
    n_requested_jobs=N_JOBS,
    engine="cpp",
    post_batch="none",
)

---
## 2. Solve exactly

Seconds per graph at N <= 21 (the solver holds 2^N floats). Not a login-node cell.

In [ ]:
from moran_process.numeric import compute

rows = {}
for g in GRAPHS:
    gc = g.to_simulation_struct()
    res = compute(gc, R)
    # u = fixation_prob(gc, R)
    rows[g.name] = {
        # "rho": u.mean(),
        "prob": res.prob.mean(), 
        # "t_abs": absorb_time(gc, R).mean(),
        "t_abs": res.time.mean(),
        # "t_fix": np.average(fixation_time(gc, R), weights=u),
        "t_fix": np.average(res.ctime, weights=res.prob), 
    }

numeric_df = pd.DataFrame(rows).T
numeric_df

In [ ]:
print(res)

---
## 3. Read the simulation

In [ ]:
fixed = pl.col("steps").filter(pl.col("fixation"))

sim_df = (
    pl.scan_parquet(SHARDS)
    .group_by("graph_name")
    .agg(
        rho=pl.col("fixation").mean(),
        t_abs=pl.col("steps").mean(),
        sd_abs=pl.col("steps").std(),
        t_fix=fixed.mean(),
        sd_fix=fixed.std(),
        n=pl.len(),
        n_fix=pl.col("fixation").sum(),
        n_censored=pl.col("censored").sum(),
    )
    .collect()
    .to_pandas()
    .set_index("graph_name")
)
assert sim_df.n_censored.sum() == 0, "censored runs bias both statistics low"
sim_df

---
## 4. Compare

`z` is (simulated - exact) in units of the simulation's own standard error, so |z| > 3 is a
real disagreement rather than Monte Carlo noise.

In [ ]:
d = numeric_df.join(sim_df, lsuffix="_num", rsuffix="_sim")

se = {
    "rho": np.sqrt(d.rho_sim * (1 - d.rho_sim) / d.n),
    "t_abs": d.sd_abs / np.sqrt(d.n),
    "t_fix": d.sd_fix / np.sqrt(d.n_fix),
}

cmp_df = pd.concat(
    {
        k: pd.DataFrame(
            {
                "exact": d[f"{k}_num"],
                "sim": d[f"{k}_sim"],
                "ratio": d[f"{k}_sim"] / d[f"{k}_num"],
                "z": (d[f"{k}_sim"] - d[f"{k}_num"]) / se[k],
            }
        )
        for k in se
    },
    axis=1,
)
cmp_df.style.background_gradient(
    cmap="RdYlGn_r", vmin=0, vmax=10, axis=None,
    subset=[(k, "z") for k in se], gmap=cmp_df.xs("z", level=1, axis=1).abs(),
).format("{:.4g}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, k in zip(axes, se):
    x, y = d[f"{k}_num"], d[f"{k}_sim"]
    lim = [min(x.min(), y.min()) * 0.8, max(x.max(), y.max()) * 1.25]
    ax.plot(lim, lim, "k--", lw=1)
    ax.scatter(x, y, s=60, zorder=3)
    for name in d.index:
        ax.annotate(name, (x[name], y[name]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim,
           xlabel="exact", ylabel="simulated", title=k)
fig.suptitle(f"Numeric solver vs {N_REPEATS:,} simulations per graph  (r = {R})")
plt.tight_layout()
plt.show()